# Can AI connect to existing business software?

A descriptive analysis of Generativ’s **25-system editorial catalogue**, reviewed on 16 October 2026. [Search and cite the database](https://www.generativ.co.uk/resources/software-integration-database) · [GitHub and contributions](https://github.com/Fantasmagazzle/uk-business-software-integrations).

**All connections are untested by Generativ in this release.** This analysis counts documented interfaces and review gaps. It does not measure UK adoption, software quality or successful integrations. Unknown does not mean unavailable. Original curation: CC BY 4.0.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
candidates = [Path('../data/systems.csv'), Path('data/systems.csv')]
candidates += list(Path('/kaggle/input').rglob('systems.csv')) if Path('/kaggle/input').exists() else []
path = next(p for p in candidates if p.exists())
systems = pd.read_csv(path, keep_default_na=False)
assert systems.id.is_unique
print(f'{len(systems)} systems; documentation reviewed {systems.last_checked.min()} to {systems.last_checked.max()}')
systems[['name', 'category', 'minimum_plan', 'webhook_status', 'verification_status']]


## 1. Which categories are covered?
This is a coverage check of the selected catalogue. Category counts must not be presented as market share.

In [ ]:
systems.category.value_counts().sort_values().plot.barh(color='#19364b', figsize=(9,4), title='Systems in the catalogue by category')
plt.xlabel('Number of selected systems'); plt.ylabel(''); plt.tight_layout(); plt.show()


## 2. What kind of event delivery is documented?
An HTTP webhook, a plan-restricted webhook and an event stream are different engineering choices. Unconfirmed entries require more research; they are not evidence of no support.

In [ ]:
webhooks = systems.webhook_status.value_counts().rename_axis('status').to_frame('systems')
display(webhooks)
webhooks.sort_values('systems').plot.barh(legend=False, color='#6b8f20', title='Documented event-delivery status in this catalogue')
plt.xlabel('Number of selected systems'); plt.ylabel(''); plt.tight_layout(); plt.show()


## 3. Where does scoping need more work?
Use these tables to identify missing plan evidence and connection tests. A documented API does not establish the customer’s licence or permissions.

In [ ]:
display(systems.groupby(['plan_status','verification_status']).size().rename('systems').reset_index())
display(systems.loc[systems.plan_status.eq('not_confirmed') | systems.webhook_status.eq('not_confirmed'), ['name','minimum_plan','plan_notes','webhook_status','detail_url']])
assert systems.verification_status.eq('not_tested').all(), 'Review narrative when test evidence changes'


## 4. Trace a capability to its source
Join the normalised tables instead of dropping provenance. `true` is vendor-documented support within the notes, not an executed test. `unknown` must remain separate from `false`.

In [ ]:
capabilities = pd.read_csv(path.parent / 'capabilities.csv', keep_default_na=False)
sources = pd.read_csv(path.parent / 'sources.csv', keep_default_na=False)
trace = capabilities.assign(source_id=capabilities.source_ids.str.split(' | ', regex=False)).explode('source_id')
trace = trace.merge(sources, left_on=['system_id','source_id'], right_on=['system_id','id'], validate='many_to_one')
display(trace.loc[trace.system_id.eq('xero'), ['entity','read','create','update','notes','title','url','checked_at']])


## Interpretation
The useful question is not simply whether a system has an API. Check the required records, plan and app eligibility, permissions, event delivery, quota scope and approval process together. These counts describe the catalogue only. Refer to each official source before implementation and test in an authorised environment.

Please submit corrections and scoped, sanitised test evidence through [GitHub](https://github.com/Fantasmagazzle/uk-business-software-integrations). Cite [Generativ’s database](https://www.generativ.co.uk/resources/software-integration-database), version 2026-10-16, when reusing the curation.